### 01 - Instalação (Bibliotecas)

In [ ]:
%pip install numpy

### 02 - Importação (Recursos)

In [ ]:
import numpy as np
import csv

print(f"Numpy Version: {np.__version__}")
print(f"\nCSV Version: {csv.__version__}")

### 03 - Leitura do CSV

In [ ]:
def load_csv_data(file_path):
  X = []
  y = []

  with open(file_path, "r", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
      passenger_survived = row["survived"]

      y.append(
        int(passenger_survived) if passenger_survived != "" else 0
      )

      passenger_class = row["pclass"]
      passenger_sex = row["sex"]
      passenger_age = row["age"]
      passenger_sibsp = row["sibsp"]
      passenger_parch = row["parch"]
      passenger_fare = row["fare"]

      X.append([
        float(passenger_class) if passenger_class != "" else 0,
        1 if (passenger_sex == "female") else 0,
        float(passenger_age) if passenger_age != "" else -1, # Será trocado pela média das idades.
        float(passenger_sibsp) if passenger_sibsp != "" else 0,
        float(passenger_parch) if passenger_parch != "" else 0,
        float(passenger_fare) if passenger_fare != "" else 0
      ])

  return np.array(X), np.array(y).reshape(-1, 1)

### 04 - Tratamento de Dados

In [ ]:
def fill_missing_age(X):
  ages = X[:, 2]

  mean_age = np.mean(ages[ages != -1])

  X[:, 2] = np.where(ages == -1, mean_age, ages)

  return X

def normalize(X):
  mean = np.mean(X, axis=0)

  std = np.std(X, axis=0)

  return (X - mean) / (std + 1e-8)

def add_bias(X):
  ones = np.ones((X.shape[0], 1))

  return np.hstack((ones, X))

### 05 - Funções do Modelo

In [ ]:
def sigmoid(z):
  return 1 / (1 + np.exp(-z))

def compute_cost(X, y, beta):
  m = len(y)

  h = sigmoid(X @ beta)

  epsilson = 1e-8

  cost = -(1 / m) * np.sum(
    (y * np.log(h + epsilson)) + ((1 - y) * np.log(1 - h + epsilson))
  )

  return cost

def gradient_descent(X, y, beta, lr, epochs):
  m = len(y)

  for i in range(0, epochs):
    h = sigmoid(X @ beta)

    gradient = (1 / m) * (X.T @ (h - y))

    beta = beta - (lr * gradient)

    if i % 100 == 0:
      cost = compute_cost(X, y, beta)

      print(f"Custo da Época ({i}): {cost:.4f} ({(cost * 100):.2f}%)")

  return beta

### 06 - Funções de Processos

In [ ]:
def train(X, y):
  X = fill_missing_age(X)
  X = normalize(X)
  X = add_bias(X)

  beta = np.zeros((X.shape[1], 1))

  beta = gradient_descent(X, y, beta, lr=0.01, epochs=1500)

  return beta, X

def predict(X, beta):
  probability = sigmoid(X @ beta)

  return (probability >= 0.5).astype(int)

def accuracy(y_true, y_predict):
  return np.mean(y_true == y_predict)

### 07 - Função de Testes

In [ ]:
def train_test_split(X, y, test_size=0.2):
  np.random.seed(42)

  indexes = np.arange(len(X))

  np.random.shuffle(indexes)

  split = int(len(X) * (1 - test_size))

  train_indexes = indexes[:split]

  test_indexes = indexes[split:]

  return X[train_indexes], X[test_indexes], y[train_indexes], y[test_indexes]

### 08 - Execução

In [ ]:
X, y = load_csv_data("./titanic.csv")

X_train, X_test, y_train, y_test = train_test_split(X, y)

beta, X_train_processed = train(X_train, y_train)

X_test = fill_missing_age(X_test)
X_test = normalize(X_test)
X_test = add_bias(X_test)

y_predict = predict(X_test, beta)

accuracy_obtained = accuracy(y_test, y_predict)

print(f"\nAcurácia: {accuracy_obtained} ({(accuracy_obtained * 100):.2f}%)")

print(f"\nPrevisões: {y_predict[:10]}")